# 5-Fold 교차 검증 및 SMOTE 적용 준비
- 03번에서 생성한 최종 학습 데이터를 불러온다.
- 학습 데이터만 5개의 fold로 나누어 교차 검증 구조를 만든다.
- 각 fold에서 학습 fold에만 SMOTE를 적용하고, 검증 fold에는 SMOTE를 적용하지 않는다.
- 베이스라인 모델 설정 및 학습은 다음 단계에서 진행한다.

### 라이브러리 로드
- 데이터 처리를 위한 pandas, numpy를 불러온다.
- 5-Fold 분할을 위해 StratifiedKFold를 사용한다.
- 클래스 불균형 보정을 위해 SMOTE를 사용한다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

### 최종 데이터 로드
- 03번에서 저장한 train / validation / test 데이터를 불러온다.
- 5-Fold와 SMOTE는 train 데이터에 대해서만 준비한다.
- validation, test 데이터는 이후 모델 비교 및 최종 평가 단계에서 사용한다.

In [2]:
train_df = pd.read_csv('csv/final_hybrid_train.csv')
val_df = pd.read_csv('csv/final_hybrid_val.csv')
test_df = pd.read_csv('csv/final_hybrid_test.csv')

print(f"Train 데이터 형태: {train_df.shape}")
print(f"Validation 데이터 형태: {val_df.shape}")
print(f"Test 데이터 형태: {test_df.shape}")

Train 데이터 형태: (6188, 147)
Validation 데이터 형태: (1326, 147)
Test 데이터 형태: (1326, 147)


### 입력 피처와 정답 라벨 분리
- `target_label`은 모델이 예측해야 할 정답이다.
- 나머지 컬럼은 PCA 임베딩과 메타데이터가 결합된 입력 피처로 사용한다.

In [3]:
target_col = 'target_label'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("Train 라벨 분포")
print(y_train.value_counts().sort_index())

print("\nValidation 라벨 분포")
print(y_val.value_counts().sort_index())

print("\nTest 라벨 분포")
print(y_test.value_counts().sort_index())

Train 라벨 분포
target_label
0    3184
1    3004
Name: count, dtype: int64

Validation 라벨 분포
target_label
0    682
1    644
Name: count, dtype: int64

Test 라벨 분포
target_label
0    682
1    644
Name: count, dtype: int64


### 5-Fold 교차 검증 설정
- 라벨 비율을 fold마다 최대한 유지하기 위해 StratifiedKFold를 사용한다.
- 각 fold는 train 데이터 안에서만 생성한다.
- validation/test 데이터는 fold 생성에 포함하지 않는다.

In [4]:
n_splits = 5

skf = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

fold_indices = list(skf.split(X_train, y_train))

print(f"생성된 fold 개수: {len(fold_indices)}")

생성된 fold 개수: 5


### 각 fold의 학습 데이터에만 SMOTE 적용
- fold별 학습 데이터에는 SMOTE를 적용한다.
- fold별 검증 데이터에는 SMOTE를 적용하지 않는다.
- 이 방식은 검증 fold 정보가 합성 샘플 생성에 섞이는 데이터 누수를 방지한다.

In [5]:
fold_datasets = []

for fold, (train_idx, valid_idx) in enumerate(fold_indices, start=1):
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]
    y_fold_valid = y_train.iloc[valid_idx]

    smote = SMOTE(random_state=42)
    X_fold_train_smote, y_fold_train_smote = smote.fit_resample(
        X_fold_train,
        y_fold_train
    )

    fold_datasets.append({
        'fold': fold,
        'X_train': X_fold_train_smote,
        'y_train': y_fold_train_smote,
        'X_valid': X_fold_valid,
        'y_valid': y_fold_valid
    })

    print(f"[Fold {fold}]")
    print("SMOTE 적용 전 학습 라벨 분포")
    print(y_fold_train.value_counts().sort_index())
    print("SMOTE 적용 후 학습 라벨 분포")
    print(pd.Series(y_fold_train_smote).value_counts().sort_index())
    print("검증 라벨 분포")
    print(y_fold_valid.value_counts().sort_index())
    print("-" * 40)

[Fold 1]
SMOTE 적용 전 학습 라벨 분포
target_label
0    2547
1    2403
Name: count, dtype: int64
SMOTE 적용 후 학습 라벨 분포
target_label
0    2547
1    2547
Name: count, dtype: int64
검증 라벨 분포
target_label
0    637
1    601
Name: count, dtype: int64
----------------------------------------
[Fold 2]
SMOTE 적용 전 학습 라벨 분포
target_label
0    2547
1    2403
Name: count, dtype: int64
SMOTE 적용 후 학습 라벨 분포
target_label
0    2547
1    2547
Name: count, dtype: int64
검증 라벨 분포
target_label
0    637
1    601
Name: count, dtype: int64
----------------------------------------
[Fold 3]
SMOTE 적용 전 학습 라벨 분포
target_label
0    2547
1    2403
Name: count, dtype: int64
SMOTE 적용 후 학습 라벨 분포
target_label
0    2547
1    2547
Name: count, dtype: int64
검증 라벨 분포
target_label
0    637
1    601
Name: count, dtype: int64
----------------------------------------
[Fold 4]
SMOTE 적용 전 학습 라벨 분포
target_label
0    2548
1    2403
Name: count, dtype: int64
SMOTE 적용 후 학습 라벨 분포
target_label
0    2548
1    2548
Name: count, dtype: int64
검증 라벨 분포
ta

### 다음 단계
- `fold_datasets`에 fold별 SMOTE 적용 학습 데이터와 원본 검증 데이터가 저장되어 있다.
- 다음 셀부터 베이스라인 모델을 설정하고 fold별 학습/평가를 진행하면 된다.
- 여기 밑에 셀 부터 모델 설정해서 넣으면 될 것 같아요.